In [ ]:
#pip install keras-tuner
#c:/data/tuner 디렉토리를 먼저 만들고 실행합니다.
###########################
import keras_tuner as kt
from tensorflow import keras

In [ ]:
###########################
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train, x_val = x_train[:-10000], x_train[-10000:]
y_train, y_val = y_train[:-10000], y_train[-10000:]

In [ ]:
###########################
x_train = x_train.reshape((-1, 28, 28, 1)) / 255.0
x_test = x_test.reshape((-1, 28, 28, 1)) / 255.0
x_val = x_val.reshape((-1, 28, 28, 1)) / 255.0

In [ ]:
###########################
def build_model(hp):
    inputs = keras.Input(shape=(28, 28, 1))
    filters = hp.Int("filters_1", min_value=16, max_value=64, step=16)
    x = keras.layers.Conv2D(filters, 3, activation="relu")(inputs)
    x = keras.layers.MaxPooling2D(pool_size=2)(x)
    filters = hp.Int("filters_2", min_value=32, max_value=128, step=16)
    x = keras.layers.Conv2D(filters, 3, activation="relu")(x)
    x = keras.layers.MaxPooling2D(pool_size=2)(x)
    filters = hp.Int("filters_3", min_value=32, max_value=128, step=16)
    x = keras.layers.Conv2D(filters, 3, activation="relu")(x)
    x = keras.layers.MaxPooling2D(pool_size=2)(x)
    x = keras.layers.Flatten()(x)
    outputs = keras.layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    optimizer = hp.Choice("optimizer", values=["rmsprop", "adam"])
    lr = hp.Float("lr", min_value=1e-3, max_value=1e-2, sampling="log")
    if optimizer == "rmsprop":
        opt = keras.optimizers.RMSprop(learning_rate=lr)
    else:
        opt = keras.optimizers.Adam(learning_rate=lr)
    model.compile(opt, loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

In [ ]:
###########################
model = build_model(kt.HyperParameters())
model.summary()

In [ ]:
###########################
tuner = kt.RandomSearch(build_model, objective="val_accuracy", max_trials=30, seed=0, directory="C:/data/tuner", project_name="rs_v1", overwrite=True)
tuner.search_space_summary()

In [ ]:
###########################
#오래 걸림
tuner.search(x_train[:1000], y_train[:1000], batch_size=32, epochs=10, validation_data=(x_val[:1000], y_val[:1000]))

In [ ]:
###########################
tuner.results_summary()

In [ ]:
###########################
best_hps = tuner.get_best_hyperparameters(1)
best_hps[0].values

In [ ]:
###########################
best_models = tuner.get_best_models(1)
final_model = best_models[0]

In [ ]:
###########################
final_model.evaluate(x_test, y_test)